In [1]:
# Set working directory
import os
os.chdir("../../")

In [14]:
# Configure file paths
sumprom_chec_glob = "sumproms/*.gz"
prom_signals = "prom_signals"

promoter_sequences_path = "../../vovam/pipelines/new_prom_def_seqs.csv"
fimo_results_path = "fimo_results/promoters/fimo.tsv"
pfm_dir = "../pfm" #HOCOMOCO v12, available for download online

kmer_output_dir = "kmer_avg_signal_files"
meme_output_path = "metadata/top10_7mers_OL4of5_core.meme" 

## Imports

In [3]:
import glob
import re
from collections import Counter
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from Bio import motifs, pairwise2
from Bio.Seq import Seq
from scipy import stats

/home/labs/barkailab/joshuabu/miniconda3/envs/analysis2/lib/python3.13/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
/home/labs/barkailab/joshuabu/miniconda3/envs/analysis2/lib/python3.13/site-packages/Bio/pairwise2.py:1432: BiopythonWarning: Import of C module failed. Falling back to pure Python implementation. This may be slooow...
  warnings.warn(


## Helper Functions

In [4]:
# Keep only samples with at least one replicate passing the correlation cutoff
def filter_reproducible(sumprom_all: pd.DataFrame, cutoff) -> pd.DataFrame:
    df = sumprom_all.copy()
    groups = pd.Series(df.columns, index=df.columns).str.rsplit("_", n=2).str[0]

    keep = []
    for _, members in groups.groupby(groups).groups.items():
        if len(members) < 2:
            continue
        corr = df[members].corr()
        np.fill_diagonal(corr.values, np.nan)
        max_corrs = corr.max(axis=1)
        reproducible = max_corrs[max_corrs >= cutoff].index.tolist()
        keep.extend(reproducible)
    return df[keep]


# Convert replicate column names to sample names
def _base_key(s):
    p = s.split("_")
    return "_".join(p[:-2]) if len(p) >= 3 else s


# Get all replicate columns for one sample
def reps_for(key, df):
    return [c for c in df.columns if _base_key(c) == key]


# Find the signal parquet file for one replicate
def find_signal_file(rep, folders):
    for d in folders:
        p = os.path.join(d, f"{rep}_signals.gz")
        if os.path.exists(p):
            return p
    raise FileNotFoundError(rep)


# Load all replicate signal files for one sample
def load_signal_files(reps: List[str], folders: List[str]) -> List[pd.DataFrame]:
    return [pd.read_parquet(find_signal_file(r, folders)) for r in reps]


# Average replicate signal matrices
def average_replicates(dfs: List[pd.DataFrame]) -> pd.DataFrame:
    return sum(dfs) / len(dfs)


# Reverse-complement a sequence
def rc(s):
    return s.translate(str.maketrans("ACGTacgt", "TGCAtgca"))[::-1]


# Collapse each kmer to one strand-aware representation
def canon(kmer):
    k = kmer.upper()
    r = rc(k)
    return k if k <= r else r


# Pick the best motif ID for each HOCOMOCO prefix
def best_motif_by_prefix_from_fimo(fimo):
    prim_map = {"A": 0, "B": 1, "C": 2, "D": 3}
    sec_order = ("P", "B", "S")
    best = {}
    for mid in fimo["motif_id"].dropna().unique():
        pref = mid.split(".", 1)[0]
        m = re.search(r"\.([A-D])$", mid)
        prim = prim_map.get(m.group(1), 9) if m else 9
        parts = mid.split(".")
        region = parts[-2] if len(parts) >= 2 else ""
        sec = next((i for i, ch in enumerate(sec_order) if ch in region), 3)
        rank = (prim, sec)
        if pref not in best or rank < best[pref][0]:
            best[pref] = (rank, mid)
    return {k: v for k, (_, v) in best.items()}


# Map each sample name to the preferred motif ID
def motif_id_for_sample(sample: str, best_by_prefix: dict, mapping: dict):
    pref = sample.split("_", 1)[0]
    motif_pref = mapping.get(pref, pref)
    return best_by_prefix.get(motif_pref)


# Resolve the PFM path for one motif ID
def get_motif_path(motif_id: str, pfm_dir: str):
    if not motif_id:
        return None
    p1 = Path(pfm_dir) / f"{motif_id}.pfm"
    p2 = Path(pfm_dir) / f"{motif_id}.motif"
    return str(p1) if p1.exists() else (str(p2) if p2.exists() else None)


# Load a four-column PFM file as a Biopython motif object
def load_pfm(path: str):
    lines = [ln.strip() for ln in open(path) if ln.strip() and not ln.startswith(">")]
    M = np.loadtxt(lines)
    if M.shape[1] != 4:
        raise ValueError(f"Expected 4 columns (A,C,G,T), got {M.shape[1]}")
    counts = {b: M[:, i].tolist() for i, b in enumerate("ACGT")}
    return motifs.Motif(counts=counts)


# Extract a degenerate core motif from a PFM using information content
def degenerate_core_by_ic(motif, bg_freq, alpha=2.0, min_ic=0.15, min_len=5):
    total = sum(bg_freq[b] for b in "ACGT")
    q = {b: bg_freq[b] / total for b in "ACGT"}
    motif.pseudocounts = {b: alpha * q[b] for b in "ACGT"}

    P = np.vstack([motif.pwm[b] for b in "ACGT"])
    q_vec = np.array([q[b] for b in "ACGT"])
    with np.errstate(divide="ignore", invalid="ignore"):
        ic = np.nansum(P * np.log2(np.divide(P, q_vec[:, None], where=(P > 0))), axis=0)

    deg, N = str(motif.degenerate_consensus), len(motif.degenerate_consensus)
    ic[np.isnan(ic)] = -np.inf
    mask = ic >= min_ic

    if np.any(mask):
        start, end = np.where(mask)[0][[0, -1]]
    else:
        start = end = int(np.nanargmax(ic))

    while (end - start + 1) < min_len and (start > 0 or end < N - 1):
        l = ic[start - 1] if start > 0 else -np.inf
        r = ic[end + 1] if end < N - 1 else -np.inf
        if r >= l and end < N - 1:
            end += 1
        elif start > 0:
            start -= 1
        else:
            break

    return deg[start:end + 1]


IUPAC_CLASS = {
    "A": "A", "C": "C", "G": "G", "T": "T",
    "R": "AG", "Y": "CT", "S": "GC", "W": "AT",
    "K": "GT", "M": "AC",
    "B": "CGT", "D": "AGT", "H": "ACT", "V": "ACG",
    "N": "ACGT",
}


# Test whether a kmer matches a core motif in either orientation
def core_kmer_match(core, kmer, window=5, min_matches=4):
    if not core or core == "NA":
        return False

    def match(a, b):
        return b in IUPAC_CLASS.get(a, "ACGT")

    def has_overlap(x, y, w, m):
        Lx, Ly = len(x), len(y)
        for shift in range(-Ly + w, Lx - w + 1):
            for i in range(max(0, shift), min(Lx, shift + Ly) - w + 1):
                matches = sum(
                    match(x[i + j], y[i + j - shift])
                    for j in range(w)
                    if 0 <= i + j < Lx and 0 <= i + j - shift < Ly
                )
                if matches >= m:
                    return True
        return False

    c = core.upper()
    rc_core = str(Seq(core).reverse_complement())
    k = kmer.upper()

    return has_overlap(c, k, window, min_matches) or has_overlap(rc_core, k, window, min_matches)


# Align the top enriched kmers to a common strand-aware reference
def align_top_n_rc(kmer, n=10):
    hits = kmer.nlargest(n, "zscore").index.tolist()
    if not hits:
        return pd.DataFrame()

    ref = hits[0]
    aligned = {ref: list(ref)}

    for q in hits[1:]:
        q_rc = str(Seq(q).reverse_complement())

        a_f = pairwise2.align.globalms(ref, q, 1, 0, -1, -0.5, penalize_end_gaps=(False, False))[0]
        a_r = pairwise2.align.globalms(ref, q_rc, 1, 0, -1, -0.5, penalize_end_gaps=(False, False))[0]
        aln = max([a_f, a_r], key=lambda a: a.score)

        ref_aln, qry_aln = aln.seqA, aln.seqB
        aligned[q] = [b for a, b in zip(ref_aln, qry_aln) if a != "-"]

    return pd.DataFrame(aligned).T


# Build a weighted PWM from the aligned top kmers
def weighted_pwm_from_aligned(kmer, aligned, pseudocount=0.0, nonneg=True):
    bases = "ACGT"
    L = aligned.shape[1]
    pwm = pd.DataFrame(pseudocount, index=list(bases), columns=range(L))
    for name, row in aligned.iterrows():
        w = float(kmer.loc[name, "zscore"])
        if nonneg:
            w = max(w, 0.0)
        if w == 0:
            continue
        for j, b in enumerate(row):
            if b in bases:
                pwm.loc[b, j] += w
    pwm = pwm.div(pwm.sum(axis=0), axis=1).fillna(0.0)
    return pwm

## Data Loading

In [5]:
# Load all sumprom matrices
sumprom_chec_files = glob.glob(sumprom_chec_glob)
sumprom_all = pd.concat([pd.read_parquet(x) for x in sumprom_chec_files], axis=1)

# Load promoter sequences and background frequencies
prom_seqs = pd.read_csv(promoter_sequences_path, index_col=0)

all_seqs = "".join(prom_seqs["seq"].astype(str)).upper()
counts = Counter(all_seqs)
total = len(all_seqs)
yeast_prom_bg = {letter: count / total for letter, count in counts.items()}
human_genome_bg = {"A": 0.295, "G": 0.205, "C": 0.205, "T": 0.295}


# Load FIMO motif calls
fimo = pd.read_csv(fimo_results_path, sep="\t")

## Prepare Signals

In [15]:
# Keep reproducible samples and drop the same bad samples as in the original notebook
corr_cutoff = 0.895
sumprom_filtered = filter_reproducible(sumprom_all, cutoff=corr_cutoff)

# Set the signal folder and kmer-generation parameters
folder_paths = [prom_signals]
ks = [7]
win = 25
out_dir = kmer_output_dir
os.makedirs(out_dir, exist_ok=True)

seqs = prom_seqs["seq"].str.upper()
samples = sorted({_base_key(c) for c in sumprom_filtered.columns})

## Generate kmer CSV Files

In [16]:
# Compute average local signal for each strand-collapsed kmer
for sample in samples:
    try:
        print(f"Processing {sample}...")
        df = average_replicates(load_signal_files(reps_for(sample, sumprom_filtered), folder_paths))
        df = df.iloc[:, :-150] if df.shape[1] > 150 else df

        aligned = {}
        for idx, row in df.iterrows():
            a = row.to_numpy()
            m = ~np.isnan(a)
            if not m.any():
                continue
            first = int(np.argmax(m))
            aligned[idx] = a[first:]

        for k in ks:
            agg = {}
            for idx, seq in seqs.items():
                if idx not in aligned:
                    continue
                a = aligned[idx][:len(seq)]
                for i in range(len(seq) - k + 1):
                    key = canon(seq[i:i + k])
                    center = i + k // 2
                    s = max(0, center - win)
                    e = min(len(a), center + win + 1)
                    v = float(np.nanmean(a[s:e]))
                    if not np.isnan(v):
                        agg.setdefault(key, []).append(v)

            out = pd.DataFrame(
                [(km, np.mean(vs)) for km, vs in agg.items()],
                columns=["kmer", "avg_signal"],
            ).sort_values("avg_signal", ascending=False)
            out.to_csv(os.path.join(out_dir, f"{sample}___{k}mer_avg_signal.csv"), index=False)

    except Exception as e:
        print(f"Skipping {sample} due to error: {e}")
        continue

Processing ATF1...



KeyboardInterrupt



## Create MEME File

In [9]:
# Map each sample to the motif family used for the core filter
genesymbol_to_hoco = {
    "ATF1": "ATF1",
    "ATF2": "ATF2",
    "ATF4": "ATF4",
    "CDX2": "CDX2",
    "CDX4": "CDX4",
    "CREB1": "CREB1",
    "CREB5": "CREB5",
    "ELF1": "ELF1",
    "ELF1_DBD": "ELF1",
    "ELF1_DBD_ELF2": "ELF1",
    "ELF1_DBD_ELK1": "ELF1",
    "ELF1_DBD_ELK4": "ELF1",
    "ELF1_DBD_ERF": "ELF1",
    "ELF1_DBD_ERG": "ELF1",
    "ELF1_DBD_FLI1": "ELF1",
    "ELF2": "ELF2",
    "ELK1": "ELK1",
    "ELK4": "ELK4",
    "ERF1": "ERF",
    "ERG": "ERG",
    "ERG_DBD": "ERG",
    "ERG_DBD_ELF1": "ERG",
    "ERG_DBD_ELF2": "ERG",
    "ERG_DBD_ELK1": "ERG",
    "ERG_DBD_ELK4": "ERG",
    "ERG_DBD_ERF": "ERG",
    "ERG_DBD_FLI1": "ERG",
    "FLI1": "FLI1",
    "FOS": "FOS",
    "FOXA2": "FOXA2",
    "FOXF1": "FOXF1",
    "FOXJ2": "FOXJ2",
    "FOXL1": "FOXL1",
    "FOXL2": "FOXL2",
    "FOXL2_DBD": "FOXL2",
    "FOXL2_DBD_FOXA2_IDR": "FOXL2",
    "FOXL2_DBD_FOXF1_IDR": "FOXL2",
    "FOXL2_DBD_FOXO3_IDR": "FOXL2",
    "FOXL2_DBD_FOXP1_IDR": "FOXL2",
    "FOXL2_DBD_FOXP2_IDR": "FOXL2",
    "FOXL2_DBD_FOXP3_IDR": "FOXL2",
    "FOXO3": "FOXO3",
    "FOXP1": "FOXP1",
    "FOXP2": "FOXP2",
    "FOXP3": "FOXP3",
    "GATA2": "GATA2",
    "GATA3": "GATA3",
    "GATA4": "GATA4",
    "GATA5": "GATA5",
    "GATA6": "GATA6",
    "HOXA10": "HXA10",
    "HOXA11": "HXA11",
    "HOXA9": "HXA9",
    "HOXB9": "HXB9",
    "HOXC10": "HXC10",
    "HOXC13": "HXC13",
    "HOXC9": "HXC9",
    "HOXD9": "HXD9",
    "MLX": "MLX",
    "MLXIPL": "MLXPL",
    "MNT": "MNT",
    "MXD4": "MAD4",
    "NFATC3": "NFAC3",
    "NFATC4": "NFAC4",
    "POU2F3": "PO2F3",
    "POU3F1": "PO3F1",
    "POU3F4": "PO3F4",
    "SOX11": "SOX11",
    "SOX13": "SOX13",
    "SOX13_DBD": "SOX13",
    "SOX13_IDR": "SOX13",
    "SOX15": "SOX15",
    "SOX17": "SOX17",
    "SOX17_DBD": "SOX17",
    "SOX17_IDR": "SOX17",
    "SOX30": "SOX30",
    "SOX30_DBD": "SOX30",
    "SOX30_IDR": "SOX30",
    "SOX4": "SOX4",
    "SOX5": "SOX5",
    "SOX5_DBD": "SOX5",
    "SOX5_IDR": "SOX5",
    "SOX6": "SOX6",
    "SOX6_DBD": "SOX6",
    "SOX6_IDR": "SOX6",
    "SOX7": "SOX7",
    "SOX7_DBD": "SOX7",
    "SOX7_IDR": "SOX7",
    "SOX9": "SOX9",
    "SOX9_DBD": "SOX9",
    "SOX9_IDR": "SOX9",
    "TGIF1": "TGIF1",
    "TGIF2": "TGIF2",
    "TGIF2LX": "TF2LX",
    "TGIF2LY": "TF2LY",
}

best_by_prefix = best_motif_by_prefix_from_fimo(fimo)
sample_to_motif = {s: motif_id_for_sample(s, best_by_prefix, genesymbol_to_hoco) for s in samples}

In [13]:
# Build the core-only MEME file from the 7mer CSVs
all_samples = list(sample_to_motif.keys())

pwm_by_sample = {}
for s in all_samples:
    df = pd.read_csv(os.path.join(kmer_output_dir, f"{s}_7mer_avg_signal.csv"), index_col=0)
    df["zscore"] = stats.zscore(df["avg_signal"], nan_policy="omit")
    core = degenerate_core_by_ic(
        load_pfm(get_motif_path(sample_to_motif[s], pfm_dir)),
        bg_freq=human_genome_bg,
        alpha=0.01,
        min_ic=1.2,
        min_len=5,
    )
    df["core"] = df.index.to_series().map(lambda km: int(core_kmer_match(core, km)))
    aligned = align_top_n_rc(df[df["core"] == 1], n=10)
    if aligned.empty:
        continue
    pwm = weighted_pwm_from_aligned(df, aligned, pseudocount=0.0)
    pwm_by_sample[s] = (pwm, aligned.shape[0])

with open(meme_output_path, "w") as f:
    f.write("MEME version 4\n\n")
    for s, (pwm, nsites) in pwm_by_sample.items():
        f.write(f"MOTIF {s}_top10_7mers\n")
        f.write(f"letter-probability matrix: alength= 4 w= {pwm.shape[1]} nsites= {nsites}\n")
        for j in range(pwm.shape[1]):
            f.write("\t".join(f"{pwm.loc[b, j]:.16f}" for b in "ACGT") + "\n")
        f.write("\n")

print(f"Wrote {len(pwm_by_sample)} motifs to {meme_output_path}")

Wrote 59 motifs to /home/labs/barkailab/joshuabu/fimo_all_motifs_all_proms/top10_7mers_OL4of5_core_tester.meme
